# Day 030 Project: Multi-Step Automation Pipeline

## What You're Building

A `Pipeline` that orchestrates a realistic multi-step automation flow:
1. **Fetch** — load or simulate source data
2. **Process** — transform the data
3. **Summarize** — generate an AI narrative of the data
4. **Report** — write a simple text report

The pipeline uses `chain_steps` to run steps in order, stops on the first failure, and produces a complete run record with an AI-generated summary via `ai_pipeline_summary`.

## Project Requirements

1. Build a `Pipeline` with at least 3 named steps
2. Call `pipeline.run()` and store the result as `result`
3. The pipeline should complete with `summary['all_ok'] == True`
4. Verify with `_run_project_checks()`

In [ ]:
import ollama
import time

## Provided: All Helper Functions

In [ ]:
import ollama
import time


import time


def run_step(name: str, fn) -> dict:
    start = time.time()
    try:
        result = fn()
        return {
            "name":       name,
            "status":     "ok",
            "result":     result,
            "error":      None,
            "duration_s": round(time.time() - start, 3),
        }
    except Exception as e:
        return {
            "name":       name,
            "status":     "error",
            "result":     None,
            "error":      str(e),
            "duration_s": round(time.time() - start, 3),
        }


def chain_steps(steps: list, stop_on_error: bool = True) -> list:
    results = []
    failed  = False
    for name, fn in steps:
        if failed and stop_on_error:
            results.append({
                "name":       name,
                "status":     "skipped",
                "result":     None,
                "error":      None,
                "duration_s": 0.0,
            })
        else:
            step_result = run_step(name, fn)
            results.append(step_result)
            if step_result["status"] == "error":
                failed = True
    return results


def summarize_run(step_results: list) -> dict:
    statuses = [s["status"] for s in step_results]
    return {
        "total":            len(step_results),
        "passed":           statuses.count("ok"),
        "failed":           statuses.count("error"),
        "skipped":          statuses.count("skipped"),
        "total_duration_s": round(
            sum(s.get("duration_s", 0.0) for s in step_results), 3
        ),
        "all_ok":           all(s == "ok" for s in statuses),
    }


def ai_pipeline_summary(step_results: list, model: str = "llama3.2") -> str:
    summary = summarize_run(step_results)
    lines = []
    for s in step_results:
        if s["status"] == "ok":
            lines.append(f"  \u2713 {s['name']} ({s['duration_s']:.3f}s)")
        elif s["status"] == "error":
            lines.append(f"  \u2717 {s['name']}: {s['error']}")
        else:
            lines.append(f"  - {s['name']}: skipped")
    run_text = (
        f"{summary['passed']}/{summary['total']} steps passed, "
        f"{summary['total_duration_s']}s total\n"
        + "\n".join(lines)
    )
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a pipeline monitor. "
                    "Summarise a workflow run in 2\u20133 sentences. Be concise."
                ),
            },
            {
                "role": "user",
                "content": f"{run_text}\n\nSummarise the run:",
            },
        ],
    )
    return response["message"]["content"]


class Pipeline:
    def __init__(
        self,
        name: str = "pipeline",
        stop_on_error: bool = True,
        model: str = "llama3.2",
    ):
        self.name          = name
        self.stop_on_error = stop_on_error
        self.model         = model
        self._steps: list  = []

    def add_step(self, name: str, fn) -> "Pipeline":
        self._steps.append((name, fn))
        return self

    def run(self) -> dict:
        step_results = chain_steps(self._steps, stop_on_error=self.stop_on_error)
        summary      = summarize_run(step_results)
        report       = ai_pipeline_summary(step_results, model=self.model)
        return {
            "name":    self.name,
            "steps":   step_results,
            "summary": summary,
            "report":  report,
        }

## Your Pipeline

Design a multi-step automation flow. Each step should be a zero-arg callable that returns its output. Use closures (inner functions or lambdas) to capture any data you need to pass between steps.

In [ ]:
# Example multi-step pipeline — customise with your own steps

# Step 1: Fetch data (simulate loading from a source)
def fetch_data():
    # Simulate fetching articles or records
    return [
        {'title': 'AI in 2026',     'words': 850},
        {'title': 'Python tips',    'words': 420},
        {'title': 'Automation now', 'words': 610},
    ]

# Step 2: Process data (filter, transform)
_raw_data = None   # will be set by fetch_data
def process_data():
    # In a real pipeline, earlier results would be shared via a context dict
    data = fetch_data()
    return [r for r in data if r['words'] >= 500]

# Step 3: Generate AI summary of the processed data
def ai_summarize():
    processed = process_data()
    prompt = 'Summarise these articles in one sentence: ' + str(processed)
    resp = ollama.chat(
        model='llama3.2',
        messages=[{'role': 'user', 'content': prompt}],
    )
    return resp['message']['content']

# Build and run the pipeline
pipeline = Pipeline(name='article_pipeline')
pipeline.add_step('fetch',     fetch_data)
pipeline.add_step('process',   process_data)
pipeline.add_step('summarize', ai_summarize)

result = pipeline.run()
print(f"Pipeline: {result['name']}")
print(f"Outcome:  {result['summary']['passed']}/{result['summary']['total']} steps passed")
print(f"All OK:   {result['summary']['all_ok']}")
print(f"\nAI Report:\n{result['report']}")

## Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: Pipeline class exists with required methods
    try:
        assert 'Pipeline' in globals()
        for m in ('add_step', 'run'):
            assert hasattr(Pipeline, m), f'missing method: {m}'
        passed += 1; print('\u2705 Check 1: Pipeline class ready')
    except Exception as e:
        print(f'\u274c Check 1: {e}')

    # Check 2: result dict exists with required keys
    try:
        assert 'result' in globals(), 'result not defined — call pipeline.run()'
        for k in ('name', 'steps', 'summary', 'report'):
            assert k in result, f"result missing key: '{k}'"
        passed += 1; print('\u2705 Check 2: result has all required keys')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: pipeline has at least 3 steps
    try:
        assert 'result' in globals()
        assert len(result['steps']) >= 3, \
            f'pipeline should have >=3 steps, got {len(result["steps"])}'
        passed += 1; print(f'\u2705 Check 3: {len(result["steps"])} steps recorded')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: all steps passed
    try:
        assert 'result' in globals()
        s = result['summary']
        assert s['all_ok'] is True, \
            f"all_ok is {s['all_ok']} — {s['failed']} step(s) failed"
        passed += 1; print('\u2705 Check 4: all_ok=True — pipeline completed cleanly')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: AI report is a non-empty string
    try:
        assert 'result' in globals()
        assert isinstance(result['report'], str) and len(result['report']) > 10, \
            f"report should be a non-empty string: {result['report']!r}"
        passed += 1; print(f'\u2705 Check 5: report is {len(result["report"])} chars')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Add a `context: dict` to Pipeline that steps can read and write, so downstream steps can consume upstream results without re-running earlier steps
- Add `dry_run()` — print what would happen without calling any fn
- Add a `stop_on_error=False` pipeline with independent steps (e.g., send Slack notification AND write log file — run both even if one fails)
- Wire in a real automation step from earlier days: scrape a URL (Day 23), parse the text (Day 9), generate a summary (Day 7), and write an XLSX report (Day 28)
- Add a `timeout_s` parameter to run_step using `threading.Timer` to cancel steps that hang